In [ ]:
# =================================================================================================
# MANUALLY LOCALISED ECG DIGITISATION — FINAL QC VERSION
# =================================================================================================
#
# PURPOSE
# -------
# The source eye-tracking ECG images contain:
#   - 3 diagnostic 12-lead rows
#   - additional long rhythm strips between/below them
#
# Automatic equal-height cropping therefore fails.
#
# This notebook asks YOU to identify:
#
#   1. THREE diagnostic row centres
#        click TOP -> MIDDLE -> BOTTOM
#
#   2. FIVE vertical column boundaries
#        click LEFT EDGE -> col1/2 -> col2/3 -> col3/4 -> RIGHT EDGE
#
# The clicks are saved, so they only have to be done once.
#
# OUTPUTS
# -------
# manual_digitization_qc/
#
#   manual_geometry.json
#   manual_geometry.csv
#
#   MANUAL_GRID_<ECG>.png
#       Shows exactly which 12 areas were used
#
#   QC3_<ECG>.png
#       LEFT  = original crop + TRUE red tracked path
#       RIGHT = extracted pseudo-waveform
#
#   manual_digitization_metrics.csv
#
#   X_gaze_manual.npy
#       shape (10, 12, 1000, 1)
#
#
# DO NOT RETRAIN THE MODEL YET.
# First visually inspect QC3_*.png.
#
# =================================================================================================


# =================================================================================================
# 0. IMPORTS + INTERACTIVE MATPLOTLIB
# =================================================================================================

import os
import glob
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from scipy.ndimage import (
    binary_opening,
    binary_dilation,
    distance_transform_edt,
    median_filter,
)

from scipy.signal import savgol_filter


warnings.filterwarnings("ignore")


# -------------------------------------------------------------------------------------------------
# Attempt to make matplotlib interactive inside Jupyter.
#
# If "widget" is unavailable, it will try an external Qt window.
# -------------------------------------------------------------------------------------------------

try:
    ip = get_ipython()

    try:
        ip.run_line_magic(
            "matplotlib",
            "widget"
        )

        print(
            "Interactive backend: widget"
        )

    except Exception:

        try:
            ip.run_line_magic(
                "matplotlib",
                "qt"
            )

            print(
                "Interactive backend: qt"
            )

        except Exception:

            print(
                "\nWARNING:"
                "\nCould not automatically activate an interactive matplotlib backend."
                "\nIf clicking does not work, run ONE of these in a separate cell:"
                "\n"
                "\n    %matplotlib widget"
                "\n"
                "\nor:"
                "\n"
                "\n    %matplotlib qt"
                "\n"
            )

except Exception:
    pass


# =================================================================================================
# 1. PATHS
# =================================================================================================

EYE_ROOT = (
    "eye_tracking/"
    "eye-tracking-dataset-for-the-12-lead-electrocardiogram-"
    "interpretation-of-medical-practitioners-and-students-1.0.0"
)

OUT_DIR = "manual_digitization_qc"

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

GEOMETRY_JSON = os.path.join(
    OUT_DIR,
    "manual_geometry.json"
)

GEOMETRY_CSV = os.path.join(
    OUT_DIR,
    "manual_geometry.csv"
)


# =================================================================================================
# 2. ECG DEFINITIONS
# =================================================================================================

ECG_KEYS = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


IMAGE_FILES = {

    "NSR":
        "Normal_Sinus_Rhythm.jpg",

    "AFib":
        "Atrial_Fibrillation.png",

    "Hyperkalemia":
        "Hyperkalemia.jpg",

    "AFlutter":
        "Atrial_Flutter.jpg",

    "VTach":
        "Ventricular_Tachycardia.jpg",

    "WPW":
        "Wolf_Parkinson_White_Syndrome.jpg",

    "VentPaced":
        "Ventricular_Paced_Rhythm.jpg",

    "LBBB":
        "Left_Bundle_Branch_Block.jpg",

    "STEMI":
        "STEMI.png",

    "CompleteHeartBlock":
        "Complete_Heart_Block.jpg",
}


# -------------------------------------------------------------------------------------------------
# Physical display order in the source ECG images
# -------------------------------------------------------------------------------------------------

DISPLAY_LEADS = [

    [
        "I",
        "aVR",
        "V1",
        "V4"
    ],

    [
        "II",
        "aVL",
        "V2",
        "V5"
    ],

    [
        "III",
        "aVF",
        "V3",
        "V6"
    ],
]


# -------------------------------------------------------------------------------------------------
# Correct order expected by the pretrained waveform network
# -------------------------------------------------------------------------------------------------

STANDARD_LEADS = [

    "I",
    "II",
    "III",

    "aVR",
    "aVL",
    "aVF",

    "V1",
    "V2",
    "V3",

    "V4",
    "V5",
    "V6",
]


# =================================================================================================
# 3. SETTINGS
# =================================================================================================

# ----------------------------------------------------------------------------------------------
# Set TRUE the first time.
#
# After you have successfully clicked all 10 ECGs, change to FALSE
# if you rerun the notebook.
# ----------------------------------------------------------------------------------------------

REDO_GEOMETRY = True


# ----------------------------------------------------------------------------------------------
# Crop height.
#
# This is a fraction of the separation between adjacent DIAGNOSTIC rows.
#
# Because the rhythm strips lie between the diagnostic rows, this must
# remain substantially less than 0.50.
# ----------------------------------------------------------------------------------------------

ROW_HALF_HEIGHT_SCALE = 0.34


# ----------------------------------------------------------------------------------------------
# Horizontal padding inside each manually defined lead column
# to remove labels / calibration lines at column starts.
# ----------------------------------------------------------------------------------------------

X_PAD_FRAC = 0.035


# ----------------------------------------------------------------------------------------------
# Working resolution for trace extraction
# ----------------------------------------------------------------------------------------------

WORK_HEIGHT = 180
WORK_WIDTH = 850

OUTPUT_SAMPLES = 1000


# ----------------------------------------------------------------------------------------------
# Tracking parameters
# ----------------------------------------------------------------------------------------------

MAX_VERTICAL_JUMP = 18

SMOOTHNESS_PENALTY = 0.12

HORIZONTAL_LINE_LENGTH = 42

VERTICAL_LINE_LENGTH = 34


# =================================================================================================
# 4. IMAGE HELPERS
# =================================================================================================

def find_image(
    filename
):

    preferred = os.path.join(
        EYE_ROOT,
        "ECGs",
        "ECG_Images",
        filename
    )

    if os.path.exists(
        preferred
    ):
        return preferred

    matches = glob.glob(
        os.path.join(
            EYE_ROOT,
            "**",
            filename
        ),
        recursive=True
    )

    if (
        len(
            matches
        )
        ==
        0
    ):
        raise FileNotFoundError(
            filename
        )

    return matches[
        0
    ]


def load_gray(
    path
):

    return (
        np.asarray(
            Image.open(
                path
            )
            .convert(
                "L"
            )
        )
        .astype(
            np.float32
        )
        /
        255.0
    )


def resize_gray(
    arr,
    width,
    height
):

    arr = np.asarray(
        arr
    )

    if (
        arr.max()
        <=
        1.0
    ):

        arr = (
            arr
            *
            255.0
        )

    arr = np.clip(
        arr,
        0,
        255
    ).astype(
        np.uint8
    )

    pil = Image.fromarray(
        arr
    )

    try:

        resample = (
            Image.Resampling.BILINEAR
        )

    except AttributeError:

        resample = (
            Image.BILINEAR
        )

    pil = pil.resize(
        (
            width,
            height
        ),
        resample=resample
    )

    return (
        np.asarray(
            pil
        )
        .astype(
            np.float32
        )
        /
        255.0
    )


# =================================================================================================
# 5. INTERACTIVE MANUAL GEOMETRY
# =================================================================================================

def collect_geometry_for_ecg(
    ecg_key,
    image_path
):

    img = load_gray(
        image_path
    )


    # ==========================================================================================
    # FIRST: THREE DIAGNOSTIC ROW CENTRES
    # ==========================================================================================

    fig, ax = plt.subplots(
        figsize=(
            18,
            10
        )
    )

    ax.imshow(
        img,
        cmap="gray"
    )

    ax.set_title(
        f"{ecg_key}\n\n"
        "CLICK THE CENTRE OF THE THREE DIAGNOSTIC 12-LEAD ROWS\n"
        "in this order: TOP → MIDDLE → BOTTOM\n\n"
        "Do NOT click the long rhythm strips.",
        fontsize=15
    )

    ax.axis(
        "off"
    )

    plt.tight_layout()

    print(
        "\n" +
        "=" * 90
    )

    print(
        ecg_key
    )

    print(
        "CLICK 3 diagnostic row centres: TOP -> MIDDLE -> BOTTOM"
    )

    row_clicks = plt.ginput(
        3,
        timeout=-1
    )

    plt.close(
        fig
    )


    if (
        len(
            row_clicks
        )
        !=
        3
    ):

        raise RuntimeError(
            f"{ecg_key}: expected 3 row clicks."
        )


    # We only need the y coordinates.
    row_centres = sorted(
        [
            float(
                p[
                    1
                ]
            )
            for p
            in row_clicks
        ]
    )


    # ==========================================================================================
    # SECOND: FIVE VERTICAL COLUMN BOUNDARIES
    # ==========================================================================================

    fig, ax = plt.subplots(
        figsize=(
            18,
            10
        )
    )

    ax.imshow(
        img,
        cmap="gray"
    )


    # Draw selected rows to help orient the user.
    for y in row_centres:

        ax.axhline(
            y,
            linewidth=1.5,
            linestyle="--"
        )


    ax.set_title(
        f"{ecg_key}\n\n"
        "CLICK FIVE VERTICAL BOUNDARIES FROM LEFT TO RIGHT:\n"
        "LEFT EDGE → I/aVR boundary → aVR/V1 boundary → V1/V4 boundary → RIGHT EDGE\n\n"
        "Click near the centre of the ECG; only the x-coordinate is used.",
        fontsize=15
    )

    ax.axis(
        "off"
    )

    plt.tight_layout()


    print(
        "CLICK 5 vertical boundaries: LEFT -> RIGHT"
    )

    boundary_clicks = plt.ginput(
        5,
        timeout=-1
    )

    plt.close(
        fig
    )


    if (
        len(
            boundary_clicks
        )
        !=
        5
    ):

        raise RuntimeError(
            f"{ecg_key}: expected 5 boundary clicks."
        )


    x_boundaries = sorted(
        [
            float(
                p[
                    0
                ]
            )
            for p
            in boundary_clicks
        ]
    )


    # ==========================================================================================
    # SANITY CHECKS
    # ==========================================================================================

    H, W = img.shape


    if not all(
        0
        <=
        y
        <
        H
        for y
        in row_centres
    ):

        raise RuntimeError(
            f"{ecg_key}: invalid row coordinates."
        )


    if not all(
        0
        <=
        x
        <
        W
        for x
        in x_boundaries
    ):

        raise RuntimeError(
            f"{ecg_key}: invalid column coordinates."
        )


    row_spacing = np.diff(
        row_centres
    )


    if (
        np.min(
            row_spacing
        )
        <
        20
    ):

        raise RuntimeError(
            f"{ecg_key}: diagnostic row clicks appear too close."
        )


    column_widths = np.diff(
        x_boundaries
    )


    if (
        np.min(
            column_widths
        )
        <
        20
    ):

        raise RuntimeError(
            f"{ecg_key}: vertical boundaries appear too close."
        )


    return {

        "row_centres":
            row_centres,

        "x_boundaries":
            x_boundaries,

        "image_width":
            int(
                W
            ),

        "image_height":
            int(
                H
            ),

        "image_path":
            image_path
    }


# =================================================================================================
# 6. LOAD OR CREATE GEOMETRY
# =================================================================================================

if (
    os.path.exists(
        GEOMETRY_JSON
    )
    and
    not REDO_GEOMETRY
):

    print(
        "\nLoading saved geometry:"
    )

    print(
        GEOMETRY_JSON
    )

    with open(
        GEOMETRY_JSON,
        "r"
    ) as f:

        geometry = json.load(
            f
        )

else:

    geometry = {}


    print(
        "\n" +
        "=" * 100
    )

    print(
        "MANUAL ECG LOCALISATION"
    )

    print(
        "=" * 100
    )

    print(
        "\nFor EACH ECG:"
    )

    print(
        "1. Click the THREE diagnostic row centres."
    )

    print(
        "2. Click the FIVE column boundaries."
    )

    print(
        "\nThe clicks will be saved automatically."
    )


    for ecg_key in ECG_KEYS:

        image_path = find_image(
            IMAGE_FILES[
                ecg_key
            ]
        )

        geometry[
            ecg_key
        ] = (
            collect_geometry_for_ecg(
                ecg_key,
                image_path
            )
        )


        # Save progressively so clicks are not lost if notebook stops.
        with open(
            GEOMETRY_JSON,
            "w"
        ) as f:

            json.dump(
                geometry,
                f,
                indent=2
            )


    print(
        "\nSaved geometry:"
    )

    print(
        GEOMETRY_JSON
    )


# =================================================================================================
# 7. EXPORT GEOMETRY AS HUMAN-READABLE CSV
# =================================================================================================

geometry_rows = []


for ecg_key in ECG_KEYS:

    g = geometry[
        ecg_key
    ]

    geometry_rows.append(
        {

            "ECG":
                ecg_key,

            "row1_y":
                g[
                    "row_centres"
                ][
                    0
                ],

            "row2_y":
                g[
                    "row_centres"
                ][
                    1
                ],

            "row3_y":
                g[
                    "row_centres"
                ][
                    2
                ],

            "x_left":
                g[
                    "x_boundaries"
                ][
                    0
                ],

            "x_col1_2":
                g[
                    "x_boundaries"
                ][
                    1
                ],

            "x_col2_3":
                g[
                    "x_boundaries"
                ][
                    2
                ],

            "x_col3_4":
                g[
                    "x_boundaries"
                ][
                    3
                ],

            "x_right":
                g[
                    "x_boundaries"
                ][
                    4
                ],

            "image_width":
                g[
                    "image_width"
                ],

            "image_height":
                g[
                    "image_height"
                ],

            "image_path":
                g[
                    "image_path"
                ],
        }
    )


pd.DataFrame(
    geometry_rows
).to_csv(
    GEOMETRY_CSV,
    index=False
)


# =================================================================================================
# 8. CALCULATE TRUE SINGLE-LEAD CROP BOXES
# =================================================================================================

def get_manual_boxes(
    ecg_key
):

    g = geometry[
        ecg_key
    ]

    rows = np.asarray(
        g[
            "row_centres"
        ],
        dtype=float
    )

    xs = np.asarray(
        g[
            "x_boundaries"
        ],
        dtype=float
    )


    # ------------------------------------------------------------------------------------------
    # Estimate diagnostic-row separation.
    #
    # Half-height is deliberately much smaller than half the row spacing
    # so the rhythm strip between rows cannot enter the crop.
    # ------------------------------------------------------------------------------------------

    row_spacing = np.median(
        np.diff(
            rows
        )
    )

    half_height = (
        ROW_HALF_HEIGHT_SCALE
        *
        row_spacing
    )


    H = int(
        g[
            "image_height"
        ]
    )

    W = int(
        g[
            "image_width"
        ]
    )


    boxes = {}


    for row_idx in range(
        3
    ):

        cy = rows[
            row_idx
        ]

        y0 = int(
            round(
                cy
                -
                half_height
            )
        )

        y1 = int(
            round(
                cy
                +
                half_height
            )
        )

        y0 = max(
            0,
            y0
        )

        y1 = min(
            H,
            y1
        )


        for col_idx in range(
            4
        ):

            lead = DISPLAY_LEADS[
                row_idx
            ][
                col_idx
            ]

            x0 = xs[
                col_idx
            ]

            x1 = xs[
                col_idx
                +
                1
            ]

            width = (
                x1
                -
                x0
            )

            x0 = (
                x0
                +
                X_PAD_FRAC
                *
                width
            )

            x1 = (
                x1
                -
                X_PAD_FRAC
                *
                width
            )

            boxes[
                lead
            ] = (

                int(
                    round(
                        x0
                    )
                ),

                int(
                    round(
                        y0
                    )
                ),

                int(
                    round(
                        x1
                    )
                ),

                int(
                    round(
                        y1
                    )
                )
            )


    return (
        boxes,
        half_height
    )


# =================================================================================================
# 9. SAVE MANUAL CROP GEOMETRY FIGURES
# =================================================================================================

def save_manual_grid(
    ecg_key
):

    path = geometry[
        ecg_key
    ][
        "image_path"
    ]

    img = load_gray(
        path
    )

    boxes, half_height = (
        get_manual_boxes(
            ecg_key
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            18,
            10
        )
    )

    ax.imshow(
        img,
        cmap="gray"
    )


    colours = plt.cm.tab20(
        np.linspace(
            0,
            1,
            12
        )
    )


    for i, lead in enumerate(
        STANDARD_LEADS
    ):

        x0, y0, x1, y1 = (
            boxes[
                lead
            ]
        )

        rect = plt.Rectangle(
            (
                x0,
                y0
            ),
            x1 - x0,
            y1 - y0,
            fill=False,
            linewidth=2.0,
            color=colours[
                i
            ]
        )

        ax.add_patch(
            rect
        )

        ax.text(
            x0 + 4,
            y0 + 18,
            lead,
            fontsize=10,
            color=colours[
                i
            ],
            bbox=dict(
                facecolor="white",
                alpha=0.8,
                edgecolor="none"
            )
        )


    ax.set_title(
        f"{ecg_key}: MANUALLY LOCALISED diagnostic lead crops",
        fontsize=16
    )

    ax.axis(
        "off"
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUT_DIR,
            f"MANUAL_GRID_{ecg_key}.png"
        ),
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )


for ecg_key in ECG_KEYS:

    save_manual_grid(
        ecg_key
    )


print(
    "\nSaved MANUAL_GRID_*.png files."
)


# =================================================================================================
# 10. ECG INK CLEANING
# =================================================================================================

def robust_image_normalisation(
    gray
):

    gray = np.asarray(
        gray,
        dtype=np.float32
    )

    low = np.percentile(
        gray,
        1
    )

    high = np.percentile(
        gray,
        99
    )


    if (
        high
        <=
        low
    ):

        return np.clip(
            gray,
            0,
            1
        )


    gray = (
        gray
        -
        low
    ) / (
        high
        -
        low
    )


    return np.clip(
        gray,
        0,
        1
    )


def clean_ecg_ink(
    gray
):

    gray = robust_image_normalisation(
        gray
    )

    darkness = (
        1.0
        -
        gray
    )


    # ------------------------------------------------------------------------------------------
    # ECG ink tends to be among darkest pixels.
    # Use a moderately restrictive threshold.
    # ------------------------------------------------------------------------------------------

    threshold = max(
        0.10,
        float(
            np.percentile(
                darkness,
                82
            )
        )
    )


    ink = (
        darkness
        >=
        threshold
    )


    # ------------------------------------------------------------------------------------------
    # Remove long horizontal grid lines
    # ------------------------------------------------------------------------------------------

    horizontal_lines = (
        binary_opening(
            ink,
            structure=np.ones(
                (
                    1,
                    HORIZONTAL_LINE_LENGTH
                ),
                dtype=bool
            )
        )
    )


    horizontal_lines = (
        binary_dilation(
            horizontal_lines,
            structure=np.ones(
                (
                    3,
                    1
                ),
                dtype=bool
            )
        )
    )


    # ------------------------------------------------------------------------------------------
    # Remove long vertical grid lines
    # ------------------------------------------------------------------------------------------

    vertical_lines = (
        binary_opening(
            ink,
            structure=np.ones(
                (
                    VERTICAL_LINE_LENGTH,
                    1
                ),
                dtype=bool
            )
        )
    )


    vertical_lines = (
        binary_dilation(
            vertical_lines,
            structure=np.ones(
                (
                    1,
                    3
                ),
                dtype=bool
            )
        )
    )


    cleaned = (
        ink
        &
        ~horizontal_lines
        &
        ~vertical_lines
    )


    return (
        darkness,
        ink,
        cleaned
    )


# =================================================================================================
# 11. CONTINUITY-CONSTRAINED VITERBI / DYNAMIC-PROGRAMMING TRACKER
# =================================================================================================

def track_trace(
    cleaned_ink,
    darkness
):

    H, W = (
        cleaned_ink.shape
    )


    # ------------------------------------------------------------------------------------------
    # Distance to closest candidate ECG-ink pixel.
    # ------------------------------------------------------------------------------------------

    distance = (
        distance_transform_edt(
            ~cleaned_ink
        )
        .astype(
            np.float32
        )
    )


    distance = np.minimum(
        distance,
        18.0
    ) / 18.0


    dark_norm = (
        darkness
        /
        (
            np.max(
                darkness
            )
            +
            1e-8
        )
    )


    # ------------------------------------------------------------------------------------------
    # Observation cost:
    #   lower near candidate ink
    #   lower at genuinely dark pixels
    # ------------------------------------------------------------------------------------------

    observation_cost = (
        distance
        -
        0.45
        *
        dark_norm
    )


    # ------------------------------------------------------------------------------------------
    # Discourage tracker from hugging crop edges.
    # ------------------------------------------------------------------------------------------

    edge_zone = max(
        5,
        int(
            0.06
            *
            H
        )
    )


    edge_penalty = np.zeros(
        H,
        dtype=np.float32
    )


    for i in range(
        edge_zone
    ):

        penalty = (
            1.5
            *
            (
                1
                -
                i
                /
                edge_zone
            )
        )

        edge_penalty[
            i
        ] += penalty

        edge_penalty[
            H
            -
            1
            -
            i
        ] += penalty


    observation_cost += (
        edge_penalty[
            :,
            None
        ]
    )


    # ------------------------------------------------------------------------------------------
    # Dynamic programming
    # ------------------------------------------------------------------------------------------

    J = (
        MAX_VERTICAL_JUMP
    )


    jumps = np.arange(
        -J,
        J + 1,
        dtype=int
    )


    previous = (
        observation_cost[
            :,
            0
        ]
        .copy()
    )


    backpointer = np.zeros(
        (
            H,
            W
        ),
        dtype=np.int16
    )


    INF = np.float32(
        1e9
    )


    all_y = np.arange(
        H,
        dtype=int
    )


    for x in range(
        1,
        W
    ):

        costs = np.full(
            (
                len(
                    jumps
                ),
                H
            ),
            INF,
            dtype=np.float32
        )


        for k, jump in enumerate(
            jumps
        ):

            source_y = (
                all_y
                -
                jump
            )


            valid = (
                (
                    source_y
                    >=
                    0
                )
                &
                (
                    source_y
                    <
                    H
                )
            )


            # Slightly nonlinear jump penalty:
            # permits genuine sharp QRS transitions but discourages
            # arbitrary transitions between grid/text structures.
            penalty = (
                SMOOTHNESS_PENALTY
                *
                (
                    abs(
                        jump
                    )
                    +
                    0.025
                    *
                    jump
                    *
                    jump
                )
            )


            costs[
                k,
                valid
            ] = (
                previous[
                    source_y[
                        valid
                    ]
                ]
                +
                penalty
            )


        best_jump_idx = np.argmin(
            costs,
            axis=0
        )


        best_previous = (
            costs[
                best_jump_idx,
                all_y
            ]
        )


        current = (
            observation_cost[
                :,
                x
            ]
            +
            best_previous
        )


        best_jumps = (
            jumps[
                best_jump_idx
            ]
        )


        backpointer[
            :,
            x
        ] = (
            all_y
            -
            best_jumps
        ).astype(
            np.int16
        )


        previous = current


    # ==========================================================================================
    # BACKTRACK
    #
    # IMPORTANT:
    # This is the TRUE IMAGE Y POSITION.
    # It is not recentered.
    # ==========================================================================================

    path_y = np.zeros(
        W,
        dtype=np.float32
    )


    path_y[
        W
        -
        1
    ] = float(
        np.argmin(
            previous
        )
    )


    for x in range(
        W
        -
        1,
        0,
        -1
    ):

        current_y = int(
            np.clip(
                round(
                    path_y[
                        x
                    ]
                ),
                0,
                H - 1
            )
        )


        path_y[
            x
            -
            1
        ] = (
            backpointer[
                current_y,
                x
            ]
        )


    # ------------------------------------------------------------------------------------------
    # Very mild de-jittering only.
    #
    # Do NOT aggressively smooth ECG morphology.
    # ------------------------------------------------------------------------------------------

    path_y = median_filter(
        path_y,
        size=3
    )


    try:

        path_y = savgol_filter(
            path_y,
            window_length=9,
            polyorder=2,
            mode="interp"
        )

    except Exception:
        pass


    return (
        path_y
        .astype(
            np.float32
        )
    )


# =================================================================================================
# 12. CONVERT TRUE Y PATH TO MODEL PSEUDO-WAVEFORM
# =================================================================================================

def path_to_signal(
    path_y
):

    path_y = np.asarray(
        path_y,
        dtype=np.float32
    )


    # Image y increases downward.
    # ECG amplitude convention increases upward.
    amplitude = (
        -path_y
    )


    # Remove baseline
    amplitude = (
        amplitude
        -
        np.median(
            amplitude
        )
    )


    old_x = np.linspace(
        0,
        1,
        len(
            amplitude
        )
    )


    new_x = np.linspace(
        0,
        1,
        OUTPUT_SAMPLES
    )


    signal = np.interp(
        new_x,
        old_x,
        amplitude
    )


    signal = (
        signal
        -
        np.mean(
            signal
        )
    )


    signal = (
        signal
        /
        (
            np.std(
                signal
            )
            +
            1e-8
        )
    )


    return (
        signal.astype(
            np.float32
        )
    )


# =================================================================================================
# 13. DIGITISE ONE ECG
# =================================================================================================

def digitise_ecg(
    ecg_key
):

    path = geometry[
        ecg_key
    ][
        "image_path"
    ]


    img = load_gray(
        path
    )


    boxes, half_height = (
        get_manual_boxes(
            ecg_key
        )
    )


    signals = {}

    debugging = {}

    metric_rows = []


    for lead in STANDARD_LEADS:

        x0, y0, x1, y1 = (
            boxes[
                lead
            ]
        )


        source_crop = img[
            y0:y1,
            x0:x1
        ]


        work_crop = resize_gray(
            source_crop,
            WORK_WIDTH,
            WORK_HEIGHT
        )


        darkness, raw_ink, cleaned_ink = (
            clean_ecg_ink(
                work_crop
            )
        )


        path_y = track_trace(
            cleaned_ink,
            darkness
        )


        signal = path_to_signal(
            path_y
        )


        signals[
            lead
        ] = signal


        debugging[
            lead
        ] = {

            "source_crop":
                source_crop,

            "work_crop":
                work_crop,

            "darkness":
                darkness,

            "raw_ink":
                raw_ink,

            "cleaned_ink":
                cleaned_ink,

            "path_y":
                path_y,

            "box":
                (
                    x0,
                    y0,
                    x1,
                    y1
                ),
        }


        steps = np.abs(
            np.diff(
                path_y
            )
        )


        second_diff = np.abs(
            np.diff(
                path_y,
                n=2
            )
        )


        # --------------------------------------------------------------------------------------
        # Path-to-ink distance:
        # a useful automatic diagnostic but NOT a replacement for visual QC.
        # --------------------------------------------------------------------------------------

        distance_to_ink = (
            distance_transform_edt(
                ~cleaned_ink
            )
        )


        path_indices = np.clip(
            np.round(
                path_y
            ).astype(
                int
            ),
            0,
            WORK_HEIGHT
            -
            1
        )


        path_distance = (
            distance_to_ink[
                path_indices,
                np.arange(
                    WORK_WIDTH
                )
            ]
        )


        metric_rows.append(
            {

                "ECG":
                    ecg_key,

                "lead":
                    lead,

                "crop_x0":
                    x0,

                "crop_y0":
                    y0,

                "crop_x1":
                    x1,

                "crop_y1":
                    y1,

                "median_vertical_step":
                    float(
                        np.median(
                            steps
                        )
                    ),

                "large_jump_fraction":
                    float(
                        np.mean(
                            steps
                            >
                            8
                        )
                    ),

                "second_difference_roughness":
                    float(
                        np.mean(
                            second_diff
                        )
                    ),

                "median_distance_to_candidate_ink":
                    float(
                        np.median(
                            path_distance
                        )
                    ),

                "mean_distance_to_candidate_ink":
                    float(
                        np.mean(
                            path_distance
                        )
                    ),
            }
        )


    X = np.stack(
        [
            signals[
                lead
            ]
            for lead
            in STANDARD_LEADS
        ],
        axis=0
    )


    return (
        X[
            ...,
            None
        ].astype(
            np.float32
        ),
        debugging,
        metric_rows
    )


# =================================================================================================
# 14. CORRECT QC FIGURE
# =================================================================================================

def save_qc3(
    ecg_key,
    X,
    debugging
):

    fig = plt.figure(
        figsize=(
            19,
            24
        )
    )


    grid = fig.add_gridspec(
        12,
        2,
        hspace=0.62,
        width_ratios=[
            1.25,
            1
        ]
    )


    for i, lead in enumerate(
        STANDARD_LEADS
    ):

        info = debugging[
            lead
        ]


        # ======================================================================================
        # LEFT: ACTUAL WORKING CROP + TRUE RED Y PATH
        # ======================================================================================

        ax1 = fig.add_subplot(
            grid[
                i,
                0
            ]
        )


        crop = info[
            "work_crop"
        ]


        path_y = info[
            "path_y"
        ]


        ax1.imshow(
            crop,
            cmap="gray",
            aspect="auto",
            origin="upper"
        )


        ax1.plot(
            np.arange(
                len(
                    path_y
                )
            ),
            path_y,
            color="red",
            linewidth=1.6
        )


        ax1.set_ylim(
            crop.shape[
                0
            ]
            -
            1,
            0
        )


        ax1.set_xlim(
            0,
            crop.shape[
                1
            ]
            -
            1
        )


        ax1.set_title(
            f"{lead}: isolated source lead + TRUE tracked path"
        )


        ax1.set_xticks([])
        ax1.set_yticks([])


        # ======================================================================================
        # RIGHT: EXTRACTED SIGNAL
        # ======================================================================================

        ax2 = fig.add_subplot(
            grid[
                i,
                1
            ]
        )


        ax2.plot(
            X[
                i,
                :,
                0
            ],
            linewidth=1.0
        )


        ax2.axhline(
            0,
            linewidth=0.5,
            alpha=0.35
        )


        ax2.set_title(
            f"{lead}: 1000-sample pseudo-waveform"
        )


        ax2.set_xlim(
            0,
            OUTPUT_SAMPLES
            -
            1
        )


        ax2.set_xticks([])
        ax2.set_yticks([])


    fig.suptitle(
        f"{ecg_key} — manually localised ECG digitisation QC",
        fontsize=19,
        y=0.997
    )


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            OUT_DIR,
            f"QC3_{ecg_key}.png"
        ),
        dpi=170,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# =================================================================================================
# 15. DIGITISE ALL TEN ECGs
# =================================================================================================

print(
    "\n" +
    "=" * 100
)

print(
    "MANUALLY LOCALISED ECG DIGITISATION"
)

print(
    "=" * 100
)


all_X = []

all_metrics = []


for ecg_key in ECG_KEYS:

    print(
        "\nProcessing:",
        ecg_key
    )


    X, debugging, metrics = (
        digitise_ecg(
            ecg_key
        )
    )


    print(
        "Shape:",
        X.shape
    )


    all_X.append(
        X
    )


    all_metrics.extend(
        metrics
    )


    save_qc3(
        ecg_key,
        X,
        debugging
    )


# =================================================================================================
# 16. FINAL ARRAY
# =================================================================================================

X_gaze_manual = np.stack(
    all_X
).astype(
    np.float32
)


assert (
    X_gaze_manual.shape
    ==
    (
        10,
        12,
        1000,
        1
    )
)


np.save(
    os.path.join(
        OUT_DIR,
        "X_gaze_manual.npy"
    ),
    X_gaze_manual
)


print(
    "\nFINAL ARRAY:",
    X_gaze_manual.shape
)


# =================================================================================================
# 17. SAVE METRICS
# =================================================================================================

metrics_df = pd.DataFrame(
    all_metrics
)


# Automatic relative flags.
# These are screening flags ONLY.
metrics_df[
    "roughness_flag"
] = (
    metrics_df[
        "second_difference_roughness"
    ]
    >
    metrics_df[
        "second_difference_roughness"
    ]
    .quantile(
        0.95
    )
)


metrics_df[
    "ink_distance_flag"
] = (
    metrics_df[
        "median_distance_to_candidate_ink"
    ]
    >
    metrics_df[
        "median_distance_to_candidate_ink"
    ]
    .quantile(
        0.95
    )
)


metrics_df[
    "automatic_QC_flag"
] = (
    metrics_df[
        "roughness_flag"
    ]
    |
    metrics_df[
        "ink_distance_flag"
    ]
)


metrics_df.to_csv(
    os.path.join(
        OUT_DIR,
        "manual_digitization_metrics.csv"
    ),
    index=False
)


print(
    "\nQC METRICS SUMMARY"
)


print(
    metrics_df[
        [
            "median_vertical_step",
            "large_jump_fraction",
            "second_difference_roughness",
            "median_distance_to_candidate_ink",
            "mean_distance_to_candidate_ink",
        ]
    ]
    .describe()
    .to_string()
)


print(
    "\nAUTOMATICALLY FLAGGED LEADS"
)


flagged = metrics_df[
    metrics_df[
        "automatic_QC_flag"
    ]
]


if (
    len(
        flagged
    )
    >
    0
):

    print(
        flagged[
            [
                "ECG",
                "lead",
                "median_vertical_step",
                "second_difference_roughness",
                "median_distance_to_candidate_ink",
            ]
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "None."
    )


# =================================================================================================
# 18. FINAL OUTPUT
# =================================================================================================

print(
    "\n" +
    "=" * 100
)

print(
    "DIGITISATION FINISHED"
)

print(
    "=" * 100
)


print(
    f"""
Saved to:

    {OUT_DIR}/


FIRST inspect:

    MANUAL_GRID_<ECG>.png

Each coloured rectangle should contain ONE diagnostic lead only.
It should NOT contain an adjacent rhythm strip.


THEN inspect:

    QC3_<ECG>.png

For each lead:

    LEFT
        The RED line must directly follow the black ECG trace.
        Its vertical location is now the TRUE tracker location.

    RIGHT
        The extracted waveform should reproduce the visible morphology.


Final array:

    {OUT_DIR}/X_gaze_manual.npy

Shape:

    {X_gaze_manual.shape}


IMPORTANT:

DO NOT RETRAIN YET.

If the QC3 figures pass visual inspection, the clean training experiment
will simply replace its previous X_gaze construction with:

    X_gaze = np.load(
        "{OUT_DIR}/X_gaze_manual.npy"
    )

and we will keep every other training and validation parameter frozen.
"""
)